# 🚗 EDR Forensic Reconstruction & Crash Severity Prediction Pipeline
**Dataset:** NHTSA Event Data Recorder (CISS Datasets 2017–2024, 46,533 Records)  
**Methodology:** Strict Leakage-Free 3-Pillar Pipeline (Train-First MICE Imputation, SMOTE Oversampling, Class Weights & Focal Loss)  
**Models Benchmarked:** LightGBM, CatBoost, XGBoost, Random Forest, FT-Transformer, SAINT Transformer  
**Evaluation Protocol:** 80–20 Randomized Shuffled Train-Test Split (37,226 Train / 9,307 Test)

This notebook implements a state-of-the-art Machine Learning and Deep Learning pipeline for predicting vehicle **`CRASH_SEVERITY`** (`Deploy`, `Near-Deploy`, `Non-Deploy`) using pre-crash telemetry logged prior to impact (Speed, Throttle, Braking, Collision Object, VIN attributes, Seatbelt status). All target proxy features (e.g. `AIRBAG_DEPLOYED`) are strictly excluded to prevent data leakage.


In [ ]:
import os
import sys
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE

import catboost as cb
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 300

OUTPUT_DIR = r"edr_pipeline_output"
REPORT_CHARTS_DIR = os.path.join(r"project_report", "charts")
SAVED_MODELS_DIR = r"saved_models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_CHARTS_DIR, exist_ok=True)
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

print("Setup Completed. Environment Ready.")


In [ ]:
MASTER_PATH = os.path.join(OUTPUT_DIR, "EDR_Fused_Master_2017_2024.csv")
master_df = pd.read_csv(MASTER_PATH)

print(f"Master Dataset Loaded: {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")
print("Target Class Distribution:")
print(master_df['CRASH_SEVERITY'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))


In [ ]:
# Exclude IDs and Target Proxy Leakage Columns
drop_cols = ['CASEID', 'VEHNO', 'SOURCE_YEAR', 'CINJURED', 'CINJSEV', 'CRASH_SEVERITY', 'AIRBAG_DEPLOYED', 'BAGTYPE_MODE']
feature_cols = [c for c in master_df.columns if c not in drop_cols]

le_target = LabelEncoder()
y_all = le_target.fit_transform(master_df['CRASH_SEVERITY'])
target_classes = list(le_target.classes_)
joblib.dump(le_target, os.path.join(SAVED_MODELS_DIR, "target_encoder.joblib"))

X_all = master_df[feature_cols].copy()

cat_features = X_all.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = X_all.select_dtypes(include=[np.number]).columns.tolist()

# 1. Categorical Encoding (Filling Missing with 'Unknown')
encoders = {}
for col in cat_features:
    X_all[col] = X_all[col].fillna("Unknown").astype(str)
    le_cat = LabelEncoder()
    X_all[col] = le_cat.fit_transform(X_all[col])
    encoders[col] = le_cat
joblib.dump(encoders, os.path.join(SAVED_MODELS_DIR, "categorical_encoders.joblib"))
joblib.dump(feature_cols, os.path.join(SAVED_MODELS_DIR, "feature_columns.joblib"))

# 2. STEP 1: Split Data FIRST to Prevent Data Leakage
print("STEP 1: Performing Randomized 80-20 Train-Test Split BEFORE Imputation...")
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X_all, y_all, test_size=0.20, shuffle=True, random_state=42, stratify=y_all
)

print(f" -> Training Set: {X_train_raw.shape[0]:,} rows")
print(f" -> Testing Set:  {X_test_raw.shape[0]:,} rows")

# 3. STEP 2: FIT MICE Imputer on Training Set ONLY, then TRANSFORM Test Set
print("STEP 2: Fitting MICE Imputer on Training Set ONLY (Zero Leakage)...")
mice_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
X_train_raw[num_features] = mice_imputer.fit_transform(X_train_raw[num_features])

print("STEP 3: Transforming Test Set using Fitted MICE Imputer...")
X_test_raw[num_features] = mice_imputer.transform(X_test_raw[num_features])
joblib.dump(mice_imputer, os.path.join(SAVED_MODELS_DIR, "mice_imputer.joblib"))

# 4. STEP 4: Apply SMOTE Oversampling on Imputed 80% Training Data
print("STEP 4: Applying SMOTE Oversampling on Imputed Training Set...")
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train_raw, y_train_raw)

# 5. STEP 5: Fit Scaler on Training Set ONLY, then Transform Test Set
scaler = StandardScaler()
X_train_dl = X_train_sm.copy()
X_test_dl = X_test_raw.copy()
X_train_dl[num_features] = scaler.fit_transform(X_train_dl[num_features])
X_test_dl[num_features] = scaler.transform(X_test_dl[num_features])
joblib.dump(scaler, os.path.join(SAVED_MODELS_DIR, "scaler.joblib"))

# Clean Feature Renaming Map
feature_rename_map = {
    'PRE_PCODE_1010_IMPACT': 'Vehicle Speed at Impact',
    'PRE_PCODE_1010_MAX': 'Pre-Crash Max Speed',
    'PRE_PCODE_1010_2S': 'Vehicle Speed at -2.0s',
    'PRE_PCODE_1010_DELTA': 'Pre-Crash Speed Delta (-2s to 0s)',
    'PRE_PCODE_1020_IMPACT': 'Engine Throttle at Impact',
    'PRE_PCODE_1020_MAX': 'Pre-Crash Max Throttle %',
    'PRE_PCODE_1020_2S': 'Engine Throttle at -2.0s',
    'PRE_PCODE_1020_DELTA': 'Throttle Delta (-2s to 0s)',
    'PRE_PCODE_1030_IMPACT': 'Brake Status at Impact',
    'PRE_PCODE_1030_MAX': 'Brake Activation Flag',
    'POST_PCODE_2010_IMPACT': 'Post-Impact Speed (0s)',
    'POST_PCODE_2010_MAX': 'Post-Impact Max Speed',
    'POST_PCODE_2010_DELTA': 'Post-Impact Speed Delta',
    'OBJCONT': 'Impact Object (OBJCONT)',
    'EVENTS': 'Number of Crash Events',
    'EVENTDESC': 'Primary Event Description',
    'CDCEVENT': 'CDC Collision Code',
    'VEHICLES': 'Total Vehicles Involved',
    'VEHICLETYPE': 'Vehicle Body Type',
    'MANUFACTURERCOMMONNAME': 'Vehicle Manufacturer',
    'TRANSMISSIONSTYLE': 'Transmission Style',
    'DRIVER_BELTUSE': 'Driver Seatbelt Use',
    'DRIVER_SEX': 'Driver Gender',
    'MAX_OCC_AGE': 'Max Occupant Age',
    'CRASHTIME': 'Crash Time of Day',
    'CRASHMONTH': 'Crash Month',
    'DAYOFWEEK': 'Day of Week',
    'CONFIG': 'Crash Configuration',
    'EQUIP': 'Avoidance Equipment Present',
    'ACTIVATE': 'Avoidance Equipment Activated',
    'EDROBTAINED': 'EDR Data Obtained Flag',
    'LFBELT': 'Left Front Belt Status',
    'RFBELT': 'Right Front Belt Status',
    'MODTYPE': 'EDR Module Type',
    'FIRE': 'Vehicle Fire Flag',
    'FUELTYPE': 'Fuel System Type',
    'FUELEAK': 'Fuel Leakage Flag',
    'LATCHUSE_MODE': 'Child Seat LATCH Use'
}
X_test_renamed = X_test_raw.rename(columns=feature_rename_map)

print("Strict Leakage-Free Preprocessing Pipeline Complete.")


In [ ]:
print("Training CatBoost Classifier...")
cb_model = cb.CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6, auto_class_weights='Balanced', loss_function='MultiClass', verbose=0, random_seed=42)
cb_model.fit(X_train_sm, y_train_sm, eval_set=(X_test_raw, y_test), early_stopping_rounds=50)
cb_model.save_model(os.path.join(SAVED_MODELS_DIR, "catboost_model.cbm"))
joblib.dump(cb_model, os.path.join(SAVED_MODELS_DIR, "catboost_model.joblib"))

print("Training LightGBM Classifier...")
lgb_model = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, max_depth=6, class_weight='balanced', random_state=42, verbose=-1)
lgb_model.fit(X_train_sm, y_train_sm, eval_set=[(X_test_raw, y_test)], callbacks=[lgb.early_stopping(50, verbose=False)])
joblib.dump(lgb_model, os.path.join(SAVED_MODELS_DIR, "lightgbm_model.joblib"))

print("Training XGBoost Classifier...")
xgb_model = xgb.XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train_sm, y_train_sm, eval_set=[(X_test_raw, y_test)], verbose=False)
xgb_model.save_model(os.path.join(SAVED_MODELS_DIR, "xgboost_model.json"))
joblib.dump(xgb_model, os.path.join(SAVED_MODELS_DIR, "xgboost_model.joblib"))

print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_sm, y_train_sm)
joblib.dump(rf_model, os.path.join(SAVED_MODELS_DIR, "random_forest_model.joblib"))

# PyTorch Deep Learning Models (Focal Loss & Attention)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_tr_num = torch.tensor(X_train_dl[num_features].values, dtype=torch.float32)
x_tr_cat = torch.tensor(X_train_dl[cat_features].values, dtype=torch.long)
y_tr_t = torch.tensor(y_train_sm, dtype=torch.long)

x_te_num = torch.tensor(X_test_dl[num_features].values, dtype=torch.float32)
x_te_cat = torch.tensor(X_test_dl[cat_features].values, dtype=torch.long)
y_te_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(x_tr_num, x_tr_cat, y_tr_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(x_te_num, x_te_cat, y_te_t), batch_size=256, shuffle=False)

cat_dims = [X_all[col].nunique() + 1 for col in cat_features]

# PyTorch Focal Loss Function
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

# SAINT Transformer PyTorch Architecture
class SAINTLayer(nn.Module):
    def __init__(self, d_model=32, nhead=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Linear(128, d_model))
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        out, _ = self.attn(x, x, x)
        x = self.n1(x + out)
        return self.n2(x + self.ff(x))

class SAINTModel(nn.Module):
    def __init__(self, num_cnt, cat_dims, d_model=32):
        super().__init__()
        self.num_embeds = nn.ModuleList([nn.Linear(1, d_model) for _ in range(num_cnt)])
        self.cat_embeds = nn.ModuleList([nn.Embedding(c+1, d_model) for c in cat_dims])
        self.layer1 = SAINTLayer(d_model)
        self.layer2 = SAINTLayer(d_model)
        total = num_cnt + len(cat_dims)
        self.head = nn.Sequential(nn.Linear(total*d_model, 128), nn.ReLU(), nn.Linear(128, 3))
    def forward(self, x_num, x_cat):
        tokens = [self.num_embeds[i](x_num[:, i:i+1]).unsqueeze(1) for i in range(x_num.shape[1])]
        tokens += [self.cat_embeds[i](x_cat[:, i].long()).unsqueeze(1) for i in range(x_cat.shape[1])]
        x = torch.cat(tokens, dim=1)
        x = self.layer2(self.layer1(x))
        return self.head(x.view(x.shape[0], -1))

print("Training SAINT Transformer PyTorch Model (with Focal Loss)...")
saint_net = SAINTModel(len(num_features), cat_dims).to(device)
opt_saint = optim.AdamW(saint_net.parameters(), lr=1e-3, weight_decay=1e-4)
focal_criterion = FocalLoss(gamma=2.0)

for epoch in range(12):
    saint_net.train()
    for b_num, b_cat, b_y in train_loader:
        b_num, b_cat, b_y = b_num.to(device), b_cat.to(device), b_y.to(device)
        opt_saint.zero_grad()
        loss = focal_criterion(saint_net(b_num, b_cat), b_y)
        loss.backward()
        opt_saint.step()

saint_dir = os.path.join(SAVED_MODELS_DIR, "saint_model")
os.makedirs(saint_dir, exist_ok=True)
torch.save(saint_net.state_dict(), os.path.join(saint_dir, "saint_weights.pt"))

saint_net.eval()
all_saint_preds, all_saint_probs = [], []
with torch.no_grad():
    for b_num, b_cat, _ in test_loader:
        logits = saint_net(b_num.to(device), b_cat.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_saint_probs.extend(probs)
        all_saint_preds.extend(np.argmax(probs, axis=1))

# FT-Transformer PyTorch Architecture
class FTTransformerModel(nn.Module):
    def __init__(self, num_cnt, cat_dims, d_token=32):
        super().__init__()
        self.num_embeds = nn.ModuleList([nn.Linear(1, d_token) for _ in range(num_cnt)])
        self.cat_embeds = nn.ModuleList([nn.Embedding(c+1, d_token) for c in cat_dims])
        enc_layer = nn.TransformerEncoderLayer(d_model=d_token, nhead=4, dim_feedforward=128, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=3)
        total = num_cnt + len(cat_dims)
        self.head = nn.Sequential(nn.Linear(total*d_token, 128), nn.ReLU(), nn.Linear(128, 3))
    def forward(self, x_num, x_cat):
        tokens = [self.num_embeds[i](x_num[:, i:i+1]).unsqueeze(1) for i in range(x_num.shape[1])]
        tokens += [self.cat_embeds[i](x_cat[:, i].long()).unsqueeze(1) for i in range(x_cat.shape[1])]
        x = self.encoder(torch.cat(tokens, dim=1))
        return self.head(x.view(x.shape[0], -1))

print("Training FT-Transformer PyTorch Model (with Focal Loss)...")
ft_net = FTTransformerModel(len(num_features), cat_dims).to(device)
opt_ft = optim.AdamW(ft_net.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(12):
    ft_net.train()
    for b_num, b_cat, b_y in train_loader:
        b_num, b_cat, b_y = b_num.to(device), b_cat.to(device), b_y.to(device)
        opt_ft.zero_grad()
        loss = focal_criterion(ft_net(b_num, b_cat), b_y)
        loss.backward()
        opt_ft.step()

ft_dir = os.path.join(SAVED_MODELS_DIR, "ft_transformer_model")
os.makedirs(ft_dir, exist_ok=True)
torch.save(ft_net.state_dict(), os.path.join(ft_dir, "ft_weights.pt"))

ft_net.eval()
all_ft_preds, all_ft_probs = [], []
with torch.no_grad():
    for b_num, b_cat, _ in test_loader:
        logits = ft_net(b_num.to(device), b_cat.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_ft_probs.extend(probs)
        all_ft_preds.extend(np.argmax(probs, axis=1))

models_dict = {
    "CatBoost": (cb_model.predict(X_test_raw).flatten(), cb_model.predict_proba(X_test_raw)),
    "LightGBM": (lgb_model.predict(X_test_raw), lgb_model.predict_proba(X_test_raw)),
    "XGBoost": (xgb_model.predict(X_test_raw), xgb_model.predict_proba(X_test_raw)),
    "Random Forest": (rf_model.predict(X_test_raw), rf_model.predict_proba(X_test_raw)),
    "FT-Transformer": (np.array(all_ft_preds), np.array(all_ft_probs)),
    "SAINT Transformer": (np.array(all_saint_preds), np.array(all_saint_probs))
}

print("All 6 Models Successfully Trained & Saved.")


In [ ]:
results_list = []
for name, (pred, proba) in models_dict.items():
    acc = accuracy_score(y_test, pred)
    f1_w = f1_score(y_test, pred, average='weighted')
    f1_m = f1_score(y_test, pred, average='macro')
    prec = precision_score(y_test, pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, pred, average='weighted')
    auc_val = roc_auc_score(y_test, proba, multi_class='ovr')
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    
    near_idx = list(target_classes).index('Near-Deploy')
    near_rec = recall_score(y_test == near_idx, pred == near_idx, zero_division=0)
    
    results_list.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted F1": f1_w,
        "Macro F1": f1_m,
        "Near-Deploy Recall": near_rec,
        "Precision": prec,
        "Recall": rec,
        "AUC-ROC": auc_val,
        "MAE": mae,
        "RMSE": rmse,
        "R2-Score": r2
    })

df_results = pd.DataFrame(results_list).sort_values("Macro F1", ascending=False)
df_results.to_csv(os.path.join(OUTPUT_DIR, "model_benchmark_results.csv"), index=False)
df_results.to_csv(os.path.join(r"project_report", "model_benchmark_results.csv"), index=False)
display(df_results.style.background_gradient(cmap="Blues"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=df_results, x='Model', y='MAE', ax=axes[0], palette='Reds_d')
axes[0].set_title('Mean Absolute Error (MAE) - Lower is Better', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30)

sns.barplot(data=df_results, x='Model', y='RMSE', ax=axes[1], palette='Oranges_d')
axes[1].set_title('Root Mean Squared Error (RMSE) - Lower is Better', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)

sns.barplot(data=df_results, x='Model', y='R2-Score', ax=axes[2], palette='Blues_d')
axes[2].set_title('R² Score (Goodness of Fit) - Higher is Better', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mae_rmse_r2_comparison.png"), dpi=300)
plt.savefig(os.path.join(REPORT_CHARTS_DIR, "mae_rmse_r2_comparison.png"), dpi=300)
plt.show()


In [ ]:
print("Computing SHAP TreeExplainer on LightGBM...")
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test_renamed)

plt.figure(figsize=(10, 12))
shap.summary_plot(
    shap_values,
    X_test_renamed,
    plot_type="bar",
    class_names=target_classes,
    show=False
)
plt.title("SHAP Feature Importance - LightGBM\nEDR Parameters | NHTSA CISS 2017-2024", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_plot_stacked.png"), dpi=300)
plt.savefig(os.path.join(REPORT_CHARTS_DIR, "shap_summary_plot_stacked.png"), dpi=300)
plt.show()


In [ ]:
deploy_idx = list(target_classes).index('Deploy')
shap_values_deploy = shap_values[deploy_idx] if isinstance(shap_values, list) else shap_values[:, :, deploy_idx]

plt.figure(figsize=(10, 12))
shap.summary_plot(
    shap_values_deploy,
    X_test_renamed,
    show=False
)
plt.title("SHAP - Deploy Class Parameter Impact (LightGBM)\nNHTSA CISS 2017-2024", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("SHAP value (impact on model output)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_plot_beeswarm.png"), dpi=300)
plt.savefig(os.path.join(REPORT_CHARTS_DIR, "shap_summary_plot_beeswarm.png"), dpi=300)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (m_name, (pred, proba)) in enumerate(models_dict.items()):
    cm_raw = confusion_matrix(y_test, pred)
    cm_norm = confusion_matrix(y_test, pred, normalize='true') * 100
    
    annot = np.empty_like(cm_raw, dtype=object)
    for i in range(cm_raw.shape[0]):
        for j in range(cm_raw.shape[1]):
            annot[i, j] = f"{cm_raw[i, j]:,}\n({cm_norm[i, j]:.1f}%)"
            
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=axes[idx], xticklabels=target_classes, yticklabels=target_classes, annot_kws={"size": 10})
    axes[idx].set_title(f"{m_name} Confusion Matrix\n[Raw Counts & Row Recall %]", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Predicted Label", fontsize=10)
    axes[idx].set_ylabel("True Ground Truth Label", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices_normalized_grid.png"), dpi=300)
plt.savefig(os.path.join(REPORT_CHARTS_DIR, "confusion_matrices_normalized_grid.png"), dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
near_idx = list(target_classes).index('Near-Deploy')
for name, (pred, proba) in models_dict.items():
    precision, recall, _ = precision_recall_curve(y_test == near_idx, proba[:, near_idx])
    ap = average_precision_score(y_test == near_idx, proba[:, near_idx])
    plt.plot(recall, precision, label=f'{name} (AP = {ap:.3f})')

plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title('Precision-Recall Curves - Near-Deploy Class (2.4% Minority)', fontsize=12, fontweight='bold')
plt.legend(loc='lower left')
plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, "pr_curves_near_deploy.png"), dpi=300)
plt.savefig(os.path.join(REPORT_CHARTS_DIR, "pr_curves_near_deploy.png"), dpi=300)
plt.show()
